# FLUX.1-dev LoRA Training for Sprites (Enhanced)
This notebook uses **ai-toolkit** to fine-tune FLUX.1-dev on your sprite dataset.

### Strategy:
1. **Smoke Test**: Run a 300-step test with 20 images to verify memory (no OOM) and style.
2. **Full Run**: Run the 2000-step training on the full dataset once verified.

Optimized for Kaggle T4 (16GB VRAM) using NF4 quantization and Gradient Checkpointing.

In [ ]:
# Step 1: Install Dependencies and Fix NumPy conflict
!pip install -U -q 'numpy<2.0' 'pandas<3.0' # Force compatible versions
!pip install -q torch torchvision accelerate transformers diffusers peft bitsandbytes safetensors sentencepiece rembg
!git clone https://github.com/ostris/ai-toolkit.git /kaggle/working/ai-toolkit
%cd /kaggle/working/ai-toolkit
!git submodule update --init --recursive
!pip install -q -r requirements.txt
import sys
sys.path.insert(0, '/kaggle/working/ai-toolkit')

# Force restart instruction for user
print('\n' + '='*50)
print('IMPORTANT: If you see a NumPy error after this, please RESTART')
print('the session (Run -> Restart Session) and run again from Step 2.')
print('='*50)

In [ ]:
# Step 2: Prepare Dataset from HF
import os
import random
from datasets import load_dataset
from tqdm import tqdm
from huggingface_hub import login
from PIL import Image

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
except:
    HF_TOKEN = ''

if HF_TOKEN:
    login(token=HF_TOKEN)

DATASET_ID = 'darklord8777/sprites'
SAVE_DIR = '/kaggle/working/train_data'
os.makedirs(SAVE_DIR, exist_ok=True)

ds = load_dataset(DATASET_ID, split='train')
print("Dataset Features:", ds.features)
print(f"Total images available: {len(ds)}")

# Settings for image collection
SMOKE_TEST = True # Set to False for the full run
LIMIT = 20 if SMOKE_TEST else len(ds)

print(f'Processing {LIMIT} images...')

count = 0
for i, item in enumerate(tqdm(ds)):
    if count >= LIMIT: break
    
    try:
        img = item['image'].convert('RGB')
        img_path = os.path.join(SAVE_DIR, f'sprite_{i:04d}.png')
        txt_path = os.path.join(SAVE_DIR, f'sprite_{i:04d}.txt')
        
        img.save(img_path)
        
        # Improved diversity in caption generation
        cls_val = item.get('class', 'character')
        act_val = item.get('action', 'idle')
        view_dir = item.get('direction', 'front') # Renamed to view_dir to avoid shadowing dir()
        
        # Varying phrasing to help generalization
        templates = [
            f'a pixel art sprite of a {cls_val}, {act_val} pose, {view_dir} view, high quality, transparent background',
            f'pixel art style {cls_val} performing {act_val}, seen from the {view_dir}, clean edges, sprite sheet style',
            f'a {act_val} {cls_val} sprite, {view_dir} perspective, pixelated, video game asset'
        ]
        caption = random.choice(templates)
        
        with open(txt_path, 'w') as f:
            f.write(caption)
        count += 1
    except Exception as e:
        print(f"Skipping image {i} due to error: {e}")

# Sanity check counts
imgs = [f for f in os.listdir(SAVE_DIR) if f.endswith('.png')]
txts = [f for f in os.listdir(SAVE_DIR) if f.endswith('.txt')]
print(f"Sanity Check: {len(imgs)} images, {len(txts)} captions found.")
if len(imgs) != len(txts):
    print("WARNING: Mismatch between image and caption counts!")

In [ ]:

import yaml

def create_config(name, steps, push_to_hub=False, sample_every=250, save_every=500):
    return {
        'job': 'extension',
        'config': {
            'name': name,
            'process': [{
                'type': 'sd_trainer',
                'training_folder': '/kaggle/working/output',
                'device': 'cuda:0',
                'model': {
                    'name_or_path': 'black-forest-labs/FLUX.1-dev',
                    'is_flux': True,
                    'quantize': True,
                    'quantize_activation': True
                },
                'network': {
                    'type': 'lora',
                    'linear': 4,
                    'linear_alpha': 4
                },
                'train': {
                    'batch_size': 1,
                    'gradient_accumulation_steps': 4,
                    'gradient_checkpointing': True,
                    'steps': steps,
                    'lr': 0.0001,
                    'optimizer': 'adamw8bit',
                    'dtype': 'bf16'
                },
                'datasets': [{
                    'folder_path': '/kaggle/working/train_data',
                    'caption_ext': 'txt',
                    'resolution': [384]
                }],
                'save': {
                    'push_to_hub': push_to_hub,
                    'hf_repo_id': 'darklord8777/sprite-generator-model',
                    'hf_private': True,
                    'save_every': save_every,
                    'max_step_saves_to_keep': 3
                },
                'sample': {
                    'sampler': 'flowmatch',
                    'sample_every': sample_every,
                    'seed': 42,
                    'prompts': [
                        'a pixel art sprite of a character, idle pose, front view',
                        'a pixel art sprite of an enemy, attack pose, side view'
                    ]
                }
            }]
        }
    }

with open('/kaggle/working/smoke_test.yaml', 'w') as f:
    yaml.dump(create_config('sprite_smoke_test', 300, push_to_hub=False), f)

with open('/kaggle/working/full_run.yaml', 'w') as f:
    yaml.dump(create_config('sprite_flux_lora_full', 2000, push_to_hub=True, save_every=250), f)

print("Configs generated with linear=4, res=384")



In [ ]:
# Step 4: Run Training
import os, torch, gc
os.environ['HF_TOKEN'] = HF_TOKEN

# Clear GPU memory before starting
gc.collect()
torch.cuda.empty_cache()

# RECOMMENDED: Run smoke_test.yaml first!
!python run.py /kaggle/working/smoke_test.yaml

# ONCE VERIFIED, uncomment below for the full run:
# !python run.py /kaggle/working/full_run.yaml

## Step 5: Validation & Inference
After training, run this cell to test the trained LoRA.

In [ ]:
from diffusers import FluxPipeline
import torch
import os
from glob import glob

# Find the latest LoRA weights
lora_paths = glob("/kaggle/working/output/sprite_flux_lora_full/**/*.safetensors", recursive=True)
if not lora_paths:
    # Check smoke test output if full run hasn't been done
    lora_paths = glob("/kaggle/working/output/sprite_smoke_test/**/*.safetensors", recursive=True)

if lora_paths:
    latest_lora = sorted(lora_paths, key=os.path.getmtime)[-1]
    print(f"Loading LoRA: {latest_lora}")
    
    pipe = FluxPipeline.from_pretrained("black-forest-labs/FLUX.1-dev", torch_dtype=torch.bfloat16)
    pipe.load_lora_weights(latest_lora)
    pipe.to("cuda")

    prompts = [
        "a pixel art sprite of a character, idle pose, front view, high quality, transparent background", # On-template
        "a pixel art dragon, flying pose, side view, fiery breath, pixel art style" # Off-template
    ]

    for i, prompt in enumerate(prompts):
        image = pipe(
            prompt,
            num_inference_steps=28,
            guidance_scale=3.5,
            width=512,
            height=512,
        ).images[0]
        image.save(f"/kaggle/working/val_{i}.png")
        display(image)
else:
    print("No LoRA weights found. Complete training first.")